[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/sandbox/pharmaconet_to_pharmit.ipynb)

# From a docked complex to a Pharmit query

**Sandbox notebook - not workshop material.**

Takes the CpABC1-silymarin complex, derives a pharmacophore for the pocket with **PharmacoNet**,
and writes a **Pharmit session file** you can load at
[pharmit.csb.pitt.edu/search.html](https://pharmit.csb.pitt.edu/search.html) to screen a compound
library.

PharmacoNet ([Seo and Kim, Chemical Science 2024](https://doi.org/10.1039/D4SC04854G),
[repository](https://github.com/SeonghwanSeo/PharmacoNet), MIT licence) reads the **pocket**, not
the ligand. Silybin is used for exactly one thing: the mean of its atom coordinates, which tells
PharmacoNet where to look. That is what makes section 6 a real check rather than a circular one -
if features found from the protein alone land on silybin anyway, they are describing the pocket
rather than echoing the ligand back at us.

## Before you run this

Set the **runtime version to 2026.07** (*Runtime > Change runtime type*). PharmacoNet declares
`requires-python = ">=3.10,<3.13"` and pip enforces it. 2026.07 is Python 3.12.13, supported until
July 2027. **No GPU is needed**, but it is not instant: expect two to three minutes on one CPU
at the threshold this notebook uses.

> **Note:** this notebook runs on the blue group's **unpublished** CpABC1 model, which is not in
> this public repository. In Colab, section 4 asks you to upload two files from your local clone.

## Running it locally instead

It also runs outside Colab, which is much faster to iterate on. You need a Python 3.10 to 3.12
environment with `pharmaconet` and `py3Dmol` installed, registered as a Jupyter kernel:

```
conda create -y -n ubcedd312 python=3.12
conda activate ubcedd312
pip install "pharmaconet @ git+https://github.com/SeonghwanSeo/PharmacoNet.git" py3Dmol pandas ipykernel
python -m ipykernel install --user --name ubcedd312 --display-name "Python 3.12 (ubcedd312)"
```

Then open this file in VS Code and pick that kernel. Run it from `sandbox/`. It finds the repo
files by itself: the complex in `projects/blue/data/`, the weights in
`tools/pharmaconet/pharmaconet_weights.tar`, and it writes the session to
`projects/blue/data/downloads/`. Nothing is installed or uploaded.

## What you will do

- Install PharmacoNet and fetch its 139 MB of pretrained weights
- Derive a pharmacophore for the CpABC1 pocket
- Check the features against silybin's observed contacts
- Convert them to Pharmit's format and choose which points to search on
- View the query in the pocket, and save it for Pharmit

## 1. Inspect the runtime

Record what we have before installing anything, and set up the pass/fail ledger the later sections
write into. If the Python version is 3.13, nothing below will work and the error will look like a
PharmacoNet bug when it is really a runtime setting.

In [ ]:
import os, platform, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
# Locally the notebook sits in sandbox/, so the repo root is one level up.
ROOT = Path.cwd() if IN_COLAB else Path.cwd().parent
DATA = Path.cwd() if IN_COLAB else ROOT / "projects/blue/data"
OUT = Path.cwd() if IN_COLAB else DATA / "downloads"

RESULT = {}

def check(step, ok, detail=""):
    """Record a pass or fail for one step, print it, and return it."""
    RESULT[step] = ("PASS" if ok else "FAIL", detail)
    print(f"[{RESULT[step][0]}] {step}" + (f" - {detail}" if detail else ""))
    return ok

print("Colab image:", os.environ.get("COLAB_RELEASE_TAG", "not on Colab"))
ok = (3, 10) <= sys.version_info[:2] < (3, 13)
check("1. python >=3.10,<3.13", ok, f"{platform.python_version()} ({'Colab' if IN_COLAB else 'local'})")
if not ok:
    print("\nFIX: Runtime > Change runtime type > runtime version 2026.07,"
          "\nthen Runtime > Disconnect and delete runtime, and run this cell again.")

## 2. Install PharmacoNet

There is no PyPI release, so this installs from GitHub at a pinned commit. What comes with it is a
plain pip stack - `torch`, `numpy`, `numba`, `rdkit`, `openbabel-wheel`, `biopython`. **No DGL, no
conda, no kernel restart.**

Locally this installs nothing - it uses whatever is already in the kernel you picked.

> **Note:** we call PharmacoNet's Python API rather than its documented `modeling.py` script. That
> script imports PyMOL at the top of the file, and PyMOL is conda-only, so it cannot run here. The
> API is what `modeling.py` calls anyway.

In [ ]:
import subprocess

COMMIT = "0f2feec4e398b618fa04ce6eca8f86d2855160f4"  # main, 2025-07-15, "Finish v2.2.0"
PACKAGE = f"pharmaconet @ git+https://github.com/SeonghwanSeo/PharmacoNet.git@{COMMIT}"

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", PACKAGE, "py3Dmol"], check=True)
else:
    print("Local run: using the packages already in this kernel.")

Check that the imports which usually break come up together.

In [ ]:
import importlib.metadata as meta

try:
    import torch
    from pmnet.module import PharmacoNet
    check("2. imports", True, f"pharmaconet {meta.version('pharmaconet')} | torch {torch.__version__}")
except Exception as error:
    check("2. imports", False, f"{type(error).__name__}: {error}")

## 3. Fetch the model weights

The weights are 139 MB and live on a single Google Drive link with no mirror. Drive rate-limits
public files, and that limit is reached often
([issue #12](https://github.com/SeonghwanSeo/PharmacoNet/issues/12)), so this cell tries a mirror
on the workshop's own releases first and falls back to the authors' link.

> **Note:** if both fail, open the
> [Drive link](https://drive.google.com/file/d/1gzjdM7bD3jPm23LBcDXtkSk18nETL04p/view) in a
> browser, click *Make a copy*, and download your copy - a copy in your own Drive does not count
> against the original's quota.

In [ ]:
from pathlib import Path
import urllib.error, urllib.request

MIRROR = ("https://github.com/ersilia-os/ub-cedd-projects-workshop/"
          "releases/download/pmnet-weights-v2.2.0/pharmaconet_weights.tar")
DRIVE = "https://drive.google.com/uc?id=1gzjdM7bD3jPm23LBcDXtkSk18nETL04p"
LOCAL = ROOT / "tools/pharmaconet/pharmaconet_weights.tar"
WEIGHTS = LOCAL if LOCAL.exists() else Path("pharmaconet_weights.tar")
source = "already on disk"

if not WEIGHTS.exists():
    try:
        urllib.request.urlretrieve(MIRROR, WEIGHTS)
        source = "workshop mirror"
    except urllib.error.HTTPError:
        import gdown
        gdown.download(DRIVE, str(WEIGHTS), quiet=False)
        source = "authors' Google Drive"

size = WEIGHTS.stat().st_size / 1e6 if WEIGHTS.exists() else 0
check("3. weights", size > 1, f"{size:.0f} MB from {source}")

## 4. Load the CpABC1 target

PharmacoNet needs a protein PDB and any file whose coordinates mark the pocket. We use
`cpabc1_receptor.pdb` (protein only, 11435 heavy atoms, 1431 residues, chain A) and
`silymarin_ligand.sdf` (silybin, 35 heavy atoms, C25H22O10, same coordinate frame).

Locally these are read straight from `projects/blue/data/`. In Colab, upload them from your
clone when prompted.

In [ ]:
RECEPTOR = DATA / "cpabc1_receptor.pdb"
LIGAND = DATA / "silymarin_ligand.sdf"

if IN_COLAB and not (RECEPTOR.exists() and LIGAND.exists()):
    from google.colab import files
    print("Upload cpabc1_receptor.pdb and silymarin_ligand.sdf from projects/blue/data/")
    files.upload()

check("4. target files", RECEPTOR.exists() and LIGAND.exists(),
      f"{RECEPTOR.parent}")

PharmacoNet's pocket extraction silently drops any residue whose name is not one of the 20
standard amino acids - no warning, no error. Maestro writes `HIE`, `HID`, `HIP` for histidine and
`CYX` for a disulphide cysteine, and those would vanish without trace. Check before running.

In [ ]:
STANDARD = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split())

atoms = [line for line in open(RECEPTOR) if line.startswith(("ATOM", "HETATM"))]
odd = {line[17:20].strip() for line in atoms} - STANDARD

print(f"{len(atoms)} atoms")
check("4b. residue names", not odd, f"non-standard: {sorted(odd)}" if odd else "all standard")

## 5. Derive the pharmacophore

Two lines. `score_threshold=0.5` is deliberately lower than PharmacoNet's default.

The score is a **percentile, not a probability**: 0.9 means the hotspot scored higher than 90 per
cent of those seen in training. The default keeps only the top 15 per cent of hydrogen-bond
hotspots, which on this pocket returned ten features, nearly all hydrophobic, with no hydrogen-bond
acceptors at all. At 0.5 the polar features appear - and those are what make a Pharmit query
selective later.

> **Note:** CpABC1 is a membrane ABC transporter. PharmacoNet was trained on CrossDocked and
> Binding MOAD, which are soluble globular pockets. This target is out of distribution, so treat
> the result as a hypothesis to test, not an answer.

In [ ]:
import time

start = time.time()
module = PharmacoNet(device="cpu", weight_path=str(WEIGHTS), score_threshold=0.5, verbose=False)
model = module.run(str(RECEPTOR), ref_ligand_path=str(LIGAND))

check("5. pharmacophore", len(model.nodes) > 0,
      f"{len(model.nodes)} features in {time.time() - start:.0f} s")

Each node is one feature. `type` is what a **ligand** would need there, `center` is where
that ligand atom should sit, `hotspot_position` is the protein atom that would make the
interaction, and `radius` is the tolerance.

In [ ]:
import pandas as pd

features = pd.DataFrame([{"type": node.type, "interaction": node.interaction_type,
                          "score": round(float(node.score), 3),
                          "radius": round(float(node.radius), 2),
                          "x": node.center[0], "y": node.center[1], "z": node.center[2]}
                         for node in model.nodes])

print(features["type"].value_counts().to_string())
features.sort_values("score", ascending=False).head(10)

## 6. Check the features against silybin

PharmacoNet never saw silybin's atoms. So the question is whether features found from the protein
alone coincide with where silybin actually sits, and with the contacts it makes.

The pose's seven closest polar contacts, measured straight from the complex, are **Ser862 (2.73 A),
Asn858 (2.77), Ser888 (2.77), Thr353 (2.80), Tyr961 (2.87), Gln1101 (2.90), Arg965 (3.05)**.

Expect partial agreement, and do not read a miss as a failure. PharmacoNet models the whole pocket,
so it finds hotspots silybin never touches - which is exactly where a better molecule could gain
affinity.

In [ ]:
import numpy as np
from rdkit import Chem

ligand = Chem.MolFromMolFile(str(LIGAND))
positions = ligand.GetConformer().GetPositions()

protein = [line for line in open(RECEPTOR) if line.startswith("ATOM")]
coords = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in protein])
residues = [f"{l[17:20].strip()}{int(l[22:26])}" for l in protein]

features["to_silybin"] = [round(float(np.linalg.norm(positions - np.asarray(n.center), axis=1).min()), 2)
                          for n in model.nodes]
features["residue"] = [residues[int(np.linalg.norm(coords - np.asarray(n.hotspot_position), axis=1).argmin())]
                       for n in model.nodes]

CONTACTS = {"SER862", "ASN858", "SER888", "THR353", "TYR961", "GLN1101", "ARG965"}
found = sorted({r for r in features.residue if r in CONTACTS})
on_pose = int((features.to_silybin <= 3.0).sum())

check("6a. features on the pose", on_pose >= 3, f"{on_pose}/{len(features)} within 3.0 A of silybin")
check("6b. known contacts", len(found) >= 3, f"{len(found)}/7 recovered: {found}")

## 7. Convert to Pharmit's format

Pharmit defines exactly six feature types, in its own `src/pharmarec.cpp`. PharmacoNet has seven,
so one of them - `Halogen` - has nowhere to go and is dropped.

The rest map straight across, because both describe the **ligand-side** feature: PharmacoNet's
`HBond_donor` comes from the interaction type `HBond_ldon`, "ligand donor", which is precisely what
a Pharmit `HydrogenDonor` point means.

In [ ]:
TO_PHARMIT = {"Hydrophobic": "Hydrophobic", "Aromatic": "Aromatic",
              "Cation": "PositiveIon", "Anion": "NegativeIon",
              "HBond_donor": "HydrogenDonor", "HBond_acceptor": "HydrogenAcceptor",
              "Halogen": None}

# Pharmit's own colours and default search tolerances, from its js/pharmit.js
PHARMIT_COLOURS = {"Aromatic": "purple", "HydrogenDonor": "0xf0f0f0",
                   "HydrogenAcceptor": "orange", "Hydrophobic": "green",
                   "NegativeIon": "red", "PositiveIon": "blue"}

dropped = [n.type for n in model.nodes if not TO_PHARMIT[n.type]]
print(f"{len(model.nodes)} features, {len(dropped)} dropped ({sorted(set(dropped))})")

## 8. Choose which points to search on

This is the step that decides whether a search returns nothing, something, or everything.

Pharmit requires a hit to match **every enabled point**, so the points are ANDed together. Three
rules apply, each learned from a search that behaved badly:

- **Anchor on the pose.** Only features within 3 A of silybin may be switched on. The
  highest-scoring hydrophobic in this pocket sits 7.7 A away in a neighbouring cavity, and enabling
  it drags the search off the binding site entirely.
- **Diversity before depth.** Take the best of each feature type before a second of any type. A
  query made mostly of hydrophobic points matches almost anything; the polar features are what make
  it selective.
- **Tighten when there are too many hits.** `radius_scale` multiplies every tolerance. At 1.0 this
  query returned far too many compounds, so we use 0.7.

Everything else is written into the file but switched off, so you can toggle points in the browser
without regenerating anything.

In [ ]:
def choose(model, ligand_positions, enable=6, on_pose=3.0):
    """Return the ids of the features to switch on: on-pose, diverse, best first."""
    usable = sorted((n for n in model.nodes if TO_PHARMIT[n.type]), key=lambda n: -float(n.score))
    near = [n for n in usable
            if np.linalg.norm(ligand_positions - np.asarray(n.center), axis=1).min() <= on_pose]

    picked, seen = [], set()
    for node in near:
        if TO_PHARMIT[node.type] not in seen:
            seen.add(TO_PHARMIT[node.type])
            picked.append(id(node))
    for node in near:
        if len(picked) >= enable:
            break
        if id(node) not in picked:
            picked.append(id(node))
    return usable, set(picked[:enable])

usable, enabled = choose(model, positions, enable=6)
print(f"{len(usable)} usable points, {len(enabled)} enabled")

Now build the session. The receptor PDB is embedded in the file, which is what lets Pharmit
draw the pocket around the query - and why the file comes out around 1 MB.

In [ ]:
import json

RADIUS_SCALE = 0.7

session = {"points": [], "exselect": "receptor", "extolerance": 1, "max-hits": 25000,
           "receptor": RECEPTOR.read_text(), "recname": "cpabc1_receptor.pdb"}

for node in usable:
    x, y, z = (float(v) for v in node.center)
    session["points"].append({
        "name": TO_PHARMIT[node.type], "has_vec": False,
        "x": round(x, 3), "y": round(y, 3), "z": round(z, 3),
        "radius": round(float(node.radius) * RADIUS_SCALE, 2),
        "enabled": id(node) in enabled, "vector_on": 0,
        "svector": {"x": 1, "y": 0, "z": 0},
        "minsize": "", "maxsize": "", "selected": False})

query = pd.DataFrame([p for p in session["points"] if p["enabled"]])
query[["name", "x", "y", "z", "radius"]]

## 9. View the query in the pocket

Pharmit's viewer is built on 3Dmol, and `py3Dmol` is the same library, so this is close to what
Pharmit itself will show. The colours are Pharmit's own.

Solid spheres are the enabled points, the query Pharmit will search on. Wireframe spheres are the
features that are loaded but switched off. Silybin is in green sticks, for reference - remember
PharmacoNet did not use it to place anything.

In [ ]:
import py3Dmol

view = py3Dmol.view(width="100%", height=520)
view.addModel(session["receptor"], "pdb")
view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.55}})
view.addModel(LIGAND.read_text(), "sdf")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.15}})

for point in session["points"]:
    view.addSphere({"center": {"x": point["x"], "y": point["y"], "z": point["z"]},
                    "radius": point["radius"], "color": PHARMIT_COLOURS[point["name"]],
                    "opacity": 0.85 if point["enabled"] else 0.25,
                    "wireframe": not point["enabled"]})

view.zoomTo({"model": 1})
view.show()

## 10. Save the session for Pharmit

The file lands in `projects/blue/data/downloads/` locally, or downloads to your machine in Colab.
Load it at
[pharmit.csb.pitt.edu/search.html](https://pharmit.csb.pitt.edu/search.html) with **Load Session**.

> **Note:** the enabled set includes a `NegativeIon` point, so every hit will carry an ionisable
> acid. Silybin has no ionisable group, so this is the one place the query asks for chemistry the
> reference ligand does not have. It tightens the search hard. If you want silymarin-like hits
> instead, filter that type out before section 8 and enable a second hydrophobic in its place.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
out = OUT / "cpabc1_pharmaconet_pharmit_session.json"
out.write_text(json.dumps(session, indent=2))

check("10. pharmit session", out.stat().st_size > 1000,
      f"{out.stat().st_size / 1e6:.2f} MB, {len(query)} enabled points")

print(f"written to {out}")
if IN_COLAB:
    from google.colab import files
    files.download(str(out))

## 11. The verdict

Every section recorded a pass or a fail. Copy this block back into the chat.

In [ ]:
import datetime

print(f"PHARMACONET TO PHARMIT | {datetime.date.today()} | "
      f"image {os.environ.get('COLAB_RELEASE_TAG', '?')} | python {platform.python_version()}")
print("-" * 72)
for step, (status, detail) in RESULT.items():
    print(f"{status:4}  {step:26}  {detail}")
print("-" * 72)
print(f"{sum(s == 'PASS' for s, _ in RESULT.values())}/{len(RESULT)} passed")

## Summary

- PharmacoNet installs in Colab from a plain pip install and derives a pharmacophore for the
  CpABC1 pocket in well under a minute on CPU.
- It reads the **pocket**, not the ligand: silybin only supplies the centre of the search box. The
  section 6 comparison is therefore a genuine check, and partial agreement is the honest result.
- The pharmacophore converts cleanly to a Pharmit session, minus halogen features, which Pharmit
  does not define.
- Which points are enabled matters more than any other setting: six points, pose-anchored,
  type-diverse, at 0.7 tolerance.

**Known rough edges, all upstream:**

- `modeling.py` imports PyMOL at module scope, so the documented command line cannot run without
  conda. We use the Python API.
- No PyPI release and one misleadingly named tag, so we pin a commit.
- The weights are 139 MB behind one Google Drive link with no mirror
  ([issue #12](https://github.com/SeonghwanSeo/PharmacoNet/issues/12)).
- The pocket extraction shells out to `obabel` and ignores both its error output and its exit code.

**Next:** promote this to `projects/blue/notebooks/`, which needs the CpABC1 files committed so the
notebook can read `data/` instead of asking for an upload, and `pharmaconet` added to
`projects/blue/requirements.txt` with a `python_version < "3.13"` marker.